### Experimental Setup: Simulated Markov Process from Kernel $P$
We simulate a first-order Markov process in $\mathbb{R}^{d}$, where the transition distribution is a linear Gaussian:

$$
X_{n+1} \mid X_n = x \;\sim\; \mathcal{N}(A x + b, \Sigma)
$$


This defines a **contracting linear Gaussian** transition kernel with known stationary distribution.



### Closed-form Conditional Score

Because the conditional distribution is Gaussian, the conditional log density has a known closed-form gradient:

$$
\nabla_y \log p(y \mid x) = -\Sigma^{-1} (y - A x - b)
$$

This gives us the exact ground-truth **score function** for every transition pair $(x, y)$.


In [19]:
from tqdm.notebook import tqdm
import functools
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pdb

import functools
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import tqdm
from tqdm import tqdm
import matplotlib.pyplot as plt

import random
from torch.utils.data import TensorDataset, DataLoader

from torchvision.datasets import FashionMNIST, MNIST
from torchvision import transforms
from torch.utils.data import ConcatDataset
from torch.utils.data import Subset
from scipy import integrate
from torchvision.utils import make_grid

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
def sample_markov_chain(n_steps, A, b, Sigma, x0=None, seed=None, show_progress=False):
    """
    Generate a sample path from a Gaussian Markov chain:
        X_{t+1} | X_t ~ N(A X_t + b, Sigma)

    Parameters
    ----------
    n_steps : int
        Number of transitions (path will have length n_steps+1).
    A : np.ndarray (d x d)
        Linear transformation matrix.
    b : np.ndarray (d,)
        Bias vector.
    Sigma : np.ndarray (d x d)
        Covariance matrix (positive definite).
    x0 : np.ndarray (d,), optional
        Initial state. Defaults to zero vector.
    seed : int, optional
        Random seed for reproducibility.
    show_progress : bool, optional
        If True, show tqdm progress bar.

    Returns
    -------
    X : np.ndarray of shape (n_steps+1, d)
        The simulated Markov chain sample path.
    """
    rng = np.random.default_rng(seed)
    d = A.shape[0]
    A = np.array(A, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    Sigma = np.array(Sigma, dtype=np.float32)
    x = np.array(x0, dtype=np.float32) if x0 is not None else np.zeros_like(b, dtype=np.float32)


    # Initialize
    if x0 is None:
        x = np.zeros(d)
    else:
        x = np.array(x0)

    X = [x]
    iterator = range(n_steps)
    if show_progress:
        iterator = tqdm(iterator, desc="Simulating Markov chain")

    for _ in iterator:
        noise = rng.multivariate_normal(mean=np.zeros(d, dtype=np.float32), cov=Sigma)
        x = A @ x + b + noise
        X.append(x)

    return np.array(X)


In [8]:
# --- Example usage ---
d = 10
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

A_P = 0.8 * np.eye(d)
b_P = np.random.rand(d)
Sigma_P = 0.1 * np.eye(d)

n_pre = 100000

X = sample_markov_chain(
    n_steps=n_pre,
    A=A_P,
    b=b_P,
    Sigma=Sigma_P,
    seed=seed,
    show_progress=True  # <--- tqdm on
)

print("Full path shape:", X.shape)
torch.save(torch.from_numpy(X), "markov_path_A_0.8.pt")

Simulating Markov chain: 100%|██████████████████████████████████████| 100000/100000 [00:07<00:00, 13156.06it/s]

Full path shape: (100001, 10)


In [24]:
# Save kernel parameters to file
kernel_params = {
    "A": A_P,
    "b": b_P,
    "Sigma": Sigma_P
}
torch.save(kernel_params, "kernel_params.pt")
print("Saved kernel parameters.")


Saved kernel parameters to kernel_params.pt


In [21]:
print("Full path shape:", X.shape)
torch.save(torch.from_numpy(X), "markov_path_A_0.8.pt")

Full path shape: (100001, 10)


### Previous Approach 1: **GBRBM (Gaussian–Bernoulli Restricted Boltzmann Machine)**

- **Purpose**: Models the marginal distribution $ p(x) $
- **Assumes**: i.i.d. data $ x \sim p(x) $
- **Score learned**: $ \nabla_x \log p(x) $
- **Limitation**: Cannot model conditional transitions $ p(y \mid x) $, and not suitable for Markov chains

###  Previous Approach 2: **Denoising Score Matching (DSM)**

- **Purpose**: Learns $ \nabla_x \log p(x) $ from noisy versions of $ x $
- **Assumes**: Original data lies on a low-dimensional manifold (e.g., images)
- **Adds noise**: Artificial Gaussian noise $ x \rightarrow x + \sigma \epsilon $
- **Limitation**: My data is from a synthetic Markov process with full support in $ \mathbb{R}^d $, so noise is unnecessary and harmful

---

### Current Goal: **Conditional Score Learning**

- **Goal**: Learn $ \nabla_y \log p(y \mid x) $
- **Data**: $ (x, y) \sim \text{Markov process} $
- **Apparent Training loss**: 
  $$
  J(\theta) = \frac{1}{2} \mathbb{E}_{(x, y)} \left\| \psi_\theta(y, x) - \nabla_y \log p(y \mid x) \right\|^2
  $$
- **Actual Training loss**: 
  $$
      \tilde{J}(\theta) \triangleq \mathbb{E}_{(x, y)} \bigg( \frac{1}{2}\left\|  \psi_\theta(y, x)  \right\|^2 + \nabla_y \cdot \psi_\theta(y,x) \bigg)
  $$
  where $\nabla_y \cdot \psi_\theta(y,x)$ is the divergence of $\psi_\theta(y,x)$ w.r.t $y$, which is exactly the trace of the Jacobian of $\psi_\theta(y,x)$.
- **Advantages**:
  - I know the true conditional score (Gaussian kernel)
  - No need for noise, latent variables, or diffusion
  - Clean supervised setup, tractable and verifiable



### Conditional Score Network: Learning $\nabla_y \log p(y \mid x)$
Build a **feedforward MLP** that maps $(x, y) \in \mathbb{R}^{2d} \to \nabla_y \log p(y \mid x) \in \mathbb{R}^{d}$ and train using the loss above.
We aim to learn the conditional score function of a Markov process, i.e., the gradient of the log transition density:

$$
\psi(y, x) \approx \nabla_y \log p(y \mid x)
$$

#### Transition from Unconditional to Conditional

| Setting            | Unconditional Score Matching        | Conditional Score Matching                   |
|-------------------|-------------------------------------|----------------------------------------------|
| Input to network  | $x \in \mathbb{R}^d$              | $(y, x) \in \mathbb{R}^{2d}$                |
| Output of network | $\nabla_x \log p(x) \in \mathbb{R}^d$ | $\nabla_y \log p(y \mid x) \in \mathbb{R}^d$ |


#### Training Objective

We train the network using **score matching**, the assumption that:

- The Markov chain is **stationary** after burn-in,
- Each sample pair $(X_{n-1}, X_n)$ is drawn from the stationary joint distribution.

This framework enables us to detect changes in the Markov transition kernel by comparing learned score functions before and after the change.




### Learning and Evaluation

We train a neural network to learn the score function:

$$
\psi(y, x) \approx \nabla_y \log p(y \mid x) =s_{\text{true}}(y,x)
$$

To assess learning quality, we compute the **mean squared error (MSE)** between the trained network output and the closed-form score:

$$
\text{MSE} = \frac{1}{N} \sum_{n=1}^N \left\| \psi(X_n, X_{n-1}) + \Sigma^{-1}(X_n - A X_{n-1} - b) \right\|^2
$$

$$
  \text{VarScale} = \frac{1}{N}\sum_{n=1}^N |s_{\text{true}}(X_n,X_{n-1})|^2.
$$

If the MSE is small, i.e., relative error 
$$  
\frac{\text{MSE} }{\text{VarScale} } < 10^{-3}
$$, 
this indicates that the network has effectively learned the true conditional score.


In [9]:
#@title Define the Network
class ConditionalScoreNet(nn.Module):
    """
    Neural network to approximate conditional score:
        ψ(y, x; θ) ≈ ∇_y log p(y | x)

    Input:  concatenated vector (x, y) ∈ R^{2d}
    Output: vector in R^d
    """

    def __init__(self, d, hidden_dim=512, num_layers=5):
        super().__init__()
        layers = []

        # First layer: input = 2d, hidden_dim
        layers.append(nn.Linear(2*d, hidden_dim))
        layers.append(nn.SiLU())  # smooth activation

        # Middle layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())

        # Final layer: hidden_dim → d (score vector dimension)
        layers.append(nn.Linear(hidden_dim, d))

        # Register as Sequential
        self.net = nn.Sequential(*layers)

    def forward(self, x, y):
        """
        Forward pass.
        x: tensor of shape (batch, d)
        y: tensor of shape (batch, d)
        Returns: ψ(y, x; θ) ∈ R^{batch × d}
        """
        inp = torch.cat([x, y], dim=1)  # concatenate along features
        return self.net(inp)


In [10]:
#@title Define the loss function

def hyvarinen_loss(model, x, y):
    """
    Hyvarinen loss for conditional score learning.

    Args:
        model: ConditionalScoreNet
        x: tensor (batch, d)
        y: tensor (batch, d)

    Returns:
        scalar loss
    """
    x.requires_grad_(False)
    y.requires_grad_(True)
    psi = model(x, y)

    loss1 = 0.5 * (psi ** 2).sum(dim=1).mean()

    # Compute divergence ∇_y · ψ
    grads = []
    for i in range(psi.shape[1]):
        grad = torch.autograd.grad(psi[:, i].sum(), y, create_graph=True)[0][:, i]
        grads.append(grad)
    divergence = torch.stack(grads, dim=1).sum(dim=1)

    loss2 = divergence.mean()
    return loss1 + loss2


In [11]:
# Load path
# X = torch.load("markov_path_A0.99_b2.04_Sigma0.60.pt").float()  # shape (T, d)


# Build dataset of pairs, starting from the 50th step
x_data = X[9999:-1]   # X_{n-1}, start at index 9999
y_data = X[10000:]     # X_n, start at index 10000


print("x_data shape:", x_data.shape)  # (T-1, d)
print("y_data shape:", y_data.shape)  # (T-1, d)


x_data shape: (90001, 10)
y_data shape: (90001, 10)


In [12]:
#@title Train dataset1= markov_path_A0.80_b0_Sigma0.10.pt
# Wrap into DataLoader


dataset = TensorDataset(torch.tensor(x_data, dtype=torch.float32).to(device),
                        torch.tensor(y_data, dtype=torch.float32).to(device))


dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

# Parameters
d = X.shape[1]   # dimension of your Markov process
model = ConditionalScoreNet(d=d, hidden_dim=512, num_layers=5).to(device)  # using GPU
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

# Training loop
num_epochs = 50
# Outer loop with tqdm progress bar
for epoch in tqdm(range(num_epochs)):
    total_loss = 0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
    
        optimizer.zero_grad()
        loss = hyvarinen_loss(model, x_batch, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    tqdm.write(f"[Epoch {epoch+1}] Average Loss: {avg_loss:.6f}")


  2%|█▌                                                                         | 1/50 [01:13<59:51, 73.30s/it]

[Epoch 1] Average Loss: -21.212321


  4%|███                                                                        | 2/50 [02:20<55:40, 69.59s/it]

[Epoch 2] Average Loss: -43.471201


  6%|████▌                                                                      | 3/50 [02:30<33:20, 42.56s/it]

...

[Epoch 48] Average Loss: -49.529126


 98%|████████████████████████████████████████████████████████████████████████▌ | 49/50 [14:12<00:18, 18.04s/it]

[Epoch 49] Average Loss: -49.508282


100%|██████████████████████████████████████████████████████████████████████████| 50/50 [14:27<00:00, 17.35s/it]

[Epoch 50] Average Loss: -49.515172


In [14]:
# Save the trained model parameters in src directory
model_path = f"markov_model_A_0.8.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to markov_model_A_0.8.pth


<!-- ### Stationary Distribution

This Markov chain is geometrically ergodic and converges to its stationary distribution:

- Mean: $ \mu_\infty = (I - A_0)^{-1} b_0 = \mathbf{0} $
- Covariance:$ \Sigma_\infty = A \Sigma_\infty A^\top + \Sigma $.  Here, we have  
$
\Sigma_\infty = a^2 \Sigma_\infty + \Sigma
\Rightarrow \Sigma_\infty (1 - a^2) = \Sigmca
\Rightarrow \Sigma_\infty = \frac{\Sigma}{1 - a^2}
$


### Mixing Time

The covariance error decays as:

$$
\| \Sigma_n - \Sigma_\infty \| \le \rho(A)^{2n} \cdot \| \Sigma_0 - \Sigma_\infty \|
$$

For a tolerance of $ \varepsilon = 10^{-4} $, we solve:

$$
0.64^{2n}  \le 10^{-4} \quad \Rightarrow \quad n \ge 21
$$

Thus, we assume after **21 samples**, the data are drawn from the stationary distribution. -->

## Why Conditional Score Learning ≠ Learning $\nabla_x \log \pi(x)$

**Concern:**  
Since the Markov chain is stationary, all $X_n \sim \pi_P$.  
Does this mean the network only learns $\nabla_x \log \pi_P(x)$?

**Answer:**  
No — because the training objective uses *pairs* $(X_{n-1}, X_n)$, not singletons $X_n$.  

- The conditional score is defined by  
  $$
  \nabla_y \log p(y \mid x) = \nabla_y \log p(x,y),
  $$
  i.e. derivative of the **joint density** w.r.t. the future state $y$.
- Training on pairs with score matching aligns $\psi(x,y)$ with this conditional gradient, not the stationary score $\nabla_x \log \pi(x)$.

**Numerical evidence:**  
- I trained on stationary pairs $(x,y)$, with $n\ge 10000$.  
- I then tested on the **first 21 pre-mixing pairs** (non-stationary region).  
- Result: MSE $\approx 10^{-2}$, showing the model generalized to early transitions.  

 This confirms the network truly learned $\nabla_y \log p(y \mid x)$ (the transition dynamics), not merely $\nabla_x \log \pi(x)$ (the stationary marginal).

In [13]:

# Select test set
X_test_x = torch.tensor(X[:999], dtype=torch.float32).to(device)   # X_{n-1}
X_test_y = torch.tensor(X[1:1000], dtype=torch.float32).to(device) # X_n

N = X_test_x.shape[0]

# Convert kernel parameters to tensors
A = torch.tensor(A_P, dtype=torch.float32, device=device)
b = torch.tensor(b_P, dtype=torch.float32, device=device)
Sigma = torch.tensor(Sigma_P, dtype=torch.float32, device=device)
Sigma_inv = torch.linalg.inv(Sigma)

# Compute true score
with torch.no_grad():
    mu = X_test_x @ A.T + b  # shape (N, d)
    true_score = - (X_test_y - mu) @ Sigma_inv.T  # ∇_y log p(y|x)

# Compute predicted score
model.eval()
with torch.no_grad():
    pred_score = model(X_test_x, X_test_y)  # shape (N, d)

# === Metrics ===
# MSE = || pred - true ||^2
mse = torch.mean((pred_score - true_score)**2).item()

# VarScale = || true ||^2
varscale = torch.mean(true_score**2).item()

# Relative error
relative_error = mse / varscale

# === Report ===
print(f"MSE        = {mse:.6e}")
print(f"VarScale   = {varscale:.6e}")
print(f"Rel. Error = {relative_error:.6e}")


MSE        = 1.293524e-01
VarScale   = 1.012864e+01
Rel. Error = 1.277095e-02


In [20]:

# load model
model_path = "markov_model_A_0.8.pth"
model.load_state_dict(torch.load(model_path, map_location=device))



# Re-initialize the model exactly the same way
model = ConditionalScoreNet(d=d, hidden_dim=512, num_layers=5).to(device)

# 🔁 Load saved weights
model.load_state_dict(torch.load(model_path, map_location=device))


# Select test set
X_test_x = torch.tensor(X[:999], dtype=torch.float32).to(device)   # X_{n-1}
X_test_y = torch.tensor(X[1:1000], dtype=torch.float32).to(device) # X_n

N = X_test_x.shape[0]

# Convert kernel parameters to tensors
A = torch.tensor(A_P, dtype=torch.float32, device=device)
b = torch.tensor(b_P, dtype=torch.float32, device=device)
Sigma = torch.tensor(Sigma_P, dtype=torch.float32, device=device)
Sigma_inv = torch.linalg.inv(Sigma)

# Compute true score
with torch.no_grad():
    mu = X_test_x @ A.T + b  # shape (N, d)
    true_score = - (X_test_y - mu) @ Sigma_inv.T  # ∇_y log p(y|x)

# Compute predicted score
model.eval()
with torch.no_grad():
    pred_score = model(X_test_x, X_test_y)  # shape (N, d)

# === Metrics ===
# MSE = || pred - true ||^2
mse = torch.mean((pred_score - true_score)**2).item()

# VarScale = || true ||^2
varscale = torch.mean(true_score**2).item()

# Relative error
relative_error = mse / varscale

# === Report ===
print(f"MSE        = {mse:.6e}")
print(f"VarScale   = {varscale:.6e}")
print(f"Rel. Error = {relative_error:.6e}")


MSE        = 1.293524e-01
VarScale   = 1.012864e+01
Rel. Error = 1.277095e-02
